# Part 1: Data Acquisition, Problem Framing, and EDA

## 1. Getting the Data
We are utilizing the Scikit-Learn Diabetes dataset. The data is loaded directly into pandas DataFrames to facilitate easier Exploratory Data Analysis (EDA) and feature name tracking.

## 2. Framing the Problem
**Objective:** The goal of this project is to build a predictive model to estimate the risk of diabetes progression. Specifically, the dependent variable we are predicting is a quantitative measure of **"disease progression one year after baseline."** This model will serve as a screening tool to help physicians identify patients at high risk.

**Connecting to Course Concepts:** 
While predicting a continuous progression score is fundamentally a *Regression* problem, we can draw valuable insights from our previous workshops:
*   **Performance Metrics Classification:** In a clinical screening tool, we ultimately care about identifying "high-risk" patients. Understanding classification metrics (like Precision and Recall) helps us think about how we might eventually threshold our continuous regression predictions into actionable "High Risk" vs. "Low Risk" categories, balancing the cost of false negatives (missing a sick patient) against false positives.
*   **K-Nearest Neighbor (KNN):** The KNN workshop highlights the importance of distance metrics and feature scaling. Since our target is influenced by multiple physiological features, understanding how KNN groups similar patients reminds us that patients with similar baseline metrics should theoretically have similar progression trajectories.
*   **Logistic Regression Classifier:** Logistic regression introduces the concept of probabilistic boundaries and odds. While we are predicting a continuous number here, the underlying mathematical principles of evaluating how individual features (like BMI or blood pressure) push a patient toward a specific outcome are highly relevant to our multivariate analysis.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

# Set visualization style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# 1. Get the data
# Using as_frame=True to return pandas DataFrames, which is crucial for EDA and later feature naming
X, y = load_diabetes(return_X_y=True, as_frame=True)

# Combine into a single dataframe for easier EDA
df = X.copy()
df['progression'] = y

print("Data successfully loaded.")
print(f"Shape of X (features): {X.shape}")
print(f"Shape of y (target): {y.shape}")

In [ ]:
# 3. EDA - Statistics
print("--- Dataset Statistics ---")
display(df.describe())

# Check for missing values
print("\n--- Missing Values ---")
display(df.isnull().sum())

In [ ]:
# 3. EDA - Visualizations

# Create a figure with multiple subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Exploratory Data Analysis: Distributions and Relationships', fontsize=16)

# 1. Histogram of the Target Variable (Disease Progression)
sns.histplot(df['progression'], bins=20, kde=True, ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Distribution of Disease Progression (Target)')
axes[0, 0].set_xlabel('Progression Score')

# 2. Histogram of a key feature (BMI)
sns.histplot(df['bmi'], bins=20, kde=True, ax=axes[0, 1], color='lightgreen')
axes[0, 1].set_title('Distribution of BMI (Feature)')
axes[0, 1].set_xlabel('BMI (Scaled)')

# 3. Scatter Plot: BMI vs Target (Since BMI is used in Part 2)
sns.scatterplot(x=df['bmi'], y=df['progression'], ax=axes[1, 0], alpha=0.6, color='purple')
axes[1, 0].set_title('BMI vs. Disease Progression')
axes[1, 0].set_xlabel('BMI (Scaled)')
axes[1, 0].set_ylabel('Progression Score')

# 4. Correlation Matrix Heatmap
corr_matrix = df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool)) # Mask to hide the upper triangle
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap='coolwarm', 
            vmin=-1, vmax=1, center=0, ax=axes[1, 1], cbar_kws={"shrink": .8})
axes[1, 1].set_title('Correlation Matrix Heatmap')

plt.tight_layout()
plt.show()

## 3 & 4. EDA Insights and Data Cleaning

**Key Insights from EDA:**
1.  **Target Distribution:** The disease progression score is roughly normally distributed, centered around 150, with a range from roughly 25 to 346. 
2.  **Feature Scaling:** Looking at the descriptive statistics, the features (like `bmi`, `bp`, `s1`) have means very close to 0 and standard deviations close to 1. This indicates the dataset has **already been scaled/standardized** (mean-centered and divided by the standard deviation). 
3.  **Strong Predictors:** The correlation matrix and scatter plots reveal that `bmi` (Body Mass Index) and `s5` (likely a blood serum measurement) have the strongest positive linear correlations with disease progression (approx 0.58 and 0.57, respectively). Conversely, `bp` (blood pressure) has a very weak correlation (0.17).

**Data Cleaning:**
*   **Missing Values:** The `isnull().sum()` check confirms there are zero missing values in the dataset. 
*   **Scaling:** As noted in the insights, the data is already standardized. 
*   **Conclusion on Cleaning:** Because this is a curated Scikit-Learn toy dataset, **no data cleaning or imputation is required.** The data is ready for modeling as-is.

In [ ]:
# 5. Split the dataset into Train (75%), Validation (10%), and Test (15%)

# Step 1: Split off the Training set (75%) and a Temporary set (25%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, 
    test_size=0.25,       # 25% goes to the temp set
    random_state=42       # Set random state for reproducibility
)

# Step 2: Split the Temporary set into Validation (10%) and Test (15%)
# Math: We want 10% of the TOTAL data for Val, and 15% for Test. 
# Out of our 25% temp set, 10/25 = 40% should be Validation, and 15/25 = 60% should be Test.
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, 
    test_size=0.60,       # 60% of the 25% temp set = 15% of total data (Test)
    random_state=42
)

print("--- Dataset Split Sizes ---")
print(f"Total samples: {len(df)}")
print(f"Training set:   {len(X_train)} samples ({len(X_train)/len(df)*100:.1f}%)")
print(f"Validation set: {len(X_val)} samples ({len(X_val)/len(df)*100:.1f}%)")
print(f"Test set:       {len(X_test)} samples ({len(X_test)/len(df)*100:.1f}%)")